## Import Functions

In [1]:
import nibabel as nib
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F
from Unet import Generic_UNet, InitWeights_He # model
from utilities_for_inference import resample_by_res, resample_lb_by_shape, tta_rolling # inference

## Specifying Data and Model Path

In [2]:
PATH_FLAIR = '/mnt/controller_data/disk1/zeju/fomo-task2/preprocessed/sub_1/ses_1/flair.nii.gz'
PATH_MODEL = '/mnt/controller_data/disk1/zeju/Robust-Medical-Segmentation/output/Meningioma/3DUNet_vanilla_Meningioma_det_x320z20/model_best.pth.tar'
PATH_OUTPUT_DIR = '/mnt/controller_data/disk1/zeju/Robust-Medical-Segmentation/output/Meningioma/fomo25val/results/pred.nii.gz'
TARGET_SPACE = [0.45, 0.45, 5.0]

## Preprocess IMG

In [3]:
img_obj = sitk.ReadImage( PATH_FLAIR )
img_size_ori = img_obj.GetSize()

res_img_o = resample_by_res(img_obj, TARGET_SPACE, interpolator = sitk.sitkLinear,
                                logging = True)
# temporally save to the target mask
sitk.WriteImage(res_img_o, PATH_OUTPUT_DIR, True) 

Resampling from (384, 512, 30) to [367, 489, 34]
Spacing from (0.4296875, 0.4296875, 5.599999904632568) to [0.45, 0.45, 5.0]


In [4]:
img_obj = sitk.ReadImage( PATH_FLAIR )

array = sitk.GetArrayFromImage(img_obj)

pixel_mean = np.mean(array)
pixel_std = np.std(array)
array = (array - pixel_mean) / pixel_std

normalized_img = sitk.GetImageFromArray(array)
normalized_img.CopyInformation(img_obj)

sitk.WriteImage(normalized_img, PATH_FLAIR, True)

## Load the Model and make inference

In [5]:
## load the img
Imgloadc1 = nib.load(PATH_OUTPUT_DIR)
Imgc1 = Imgloadc1.get_fdata()
channels = Imgc1[None, ...]

In [6]:
## init the model
patch_size = [320, 320, 20]
tta = False   # True if want to use tta x 8
ttalist = [0]   # For more advanced TTA, ignore for now
ttaprob = [1]   # For more advanced TTA, ignore for now 
NumsInputChannel = 1
NumsClass = 2
# create model
conv_op = nn.Conv3d
dropout_op = nn.Dropout3d
norm_op = nn.InstanceNorm3d
conv_per_stage = 2
base_num_features = 30

norm_op_kwargs = {'eps': 1e-5, 'affine': True}
dropout_op_kwargs = {'p': 0, 'inplace': True}
net_nonlin = nn.LeakyReLU
net_nonlin_kwargs = {'negative_slope': 1e-2, 'inplace': True}
net_num_pool_op_kernel_sizes = []
for kiter in range(0, 4):  # (0,5)
    net_num_pool_op_kernel_sizes.append([2, 2, 1])
net_conv_kernel_sizes = []
for kiter in range(0, 5):  # (0,6)
    net_conv_kernel_sizes.append([3, 3, 3])

model = Generic_UNet(NumsInputChannel, base_num_features, NumsClass,
                     len(net_num_pool_op_kernel_sizes),
                     conv_per_stage, 2, conv_op, norm_op, norm_op_kwargs, dropout_op,
                     dropout_op_kwargs,
                     net_nonlin, net_nonlin_kwargs, True, False, lambda x: x, InitWeights_He(1e-2),
                     net_num_pool_op_kernel_sizes, net_conv_kernel_sizes, False, True, True)
model = model.cuda()
model.eval()

Generic_UNet(
  (conv_blocks_localization): ModuleList(
    (0): Sequential(
      (0): StackedConvLayers(
        (blocks): Sequential(
          (0): ConvDropoutNormNonlin(
            (conv): Conv3d(480, 240, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (instnorm): InstanceNorm3d(240, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
            (lrelu): LeakyReLU(negative_slope=0.01, inplace=True)
          )
        )
      )
      (1): StackedConvLayers(
        (blocks): Sequential(
          (0): ConvDropoutNormNonlin(
            (conv): Conv3d(240, 240, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
            (instnorm): InstanceNorm3d(240, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
            (lrelu): LeakyReLU(negative_slope=0.01, inplace=True)
          )
        )
      )
    )
    (1): Sequential(
      (0): StackedConvLayers(
        (blocks): Sequential(
          (0): ConvDropoutNormNonlin

In [7]:
# load model parameters
gpu_id = 0
torch.cuda.set_device(gpu_id)
checkpoint = torch.load(PATH_MODEL, map_location='cuda:' + str(gpu_id))
model.load_state_dict(checkpoint['state_dict'])

<All keys matched successfully>

In [8]:
# inference
prob = tta_rolling(model, channels, 1, patch_size, NumsInputChannel, NumsClass, tta, ttalist, ttaprob, True)
predSegmentation = np.argmax(prob, axis=0)

processing tta index:  0


In [9]:
# temporally save the results, in unifed space
npDtype = np.dtype(np.float32)
proxy_origin = nib.load(PATH_OUTPUT_DIR)
hdr_origin = proxy_origin.header
affine_origin = proxy_origin.affine
proxy_origin.uncache()

newLabelImg = nib.Nifti1Image(predSegmentation, affine_origin)
newLabelImg.set_data_dtype(npDtype)

dimsImgToSave = len(predSegmentation.shape)
newZooms = list(hdr_origin.get_zooms()[:dimsImgToSave])
if len(newZooms) < dimsImgToSave:  # Eg if original image was 3D, but I need to save a multi-channel image.
    newZooms = newZooms + [1.0] * (dimsImgToSave - len(newZooms))
newLabelImg.header.set_zooms(newZooms)

nib.save(newLabelImg, PATH_OUTPUT_DIR)

## Transform Segmentation Maps

In [10]:
seg_obj = sitk.ReadImage( PATH_OUTPUT_DIR )
res_lb_o = resample_lb_by_shape(seg_obj, [img_size_ori[2], img_size_ori[1], img_size_ori[0]], interpolator = sitk.sitkLinear,
                                  ref_img = img_obj, logging = True)
# save to the target mask
sitk.WriteImage(res_lb_o, PATH_OUTPUT_DIR, True) 

Current size (x,y,z): (367, 489, 34)
Current spacing (x,y,z): (0.44999998807907104, 0.44999998807907104, 5.0)
Target shape (z,y,x): [30, 512, 384]
Target shape (x,y,z): (384, 512, 30)
Calculated new spacing (x,y,z): [0.4300781136068205, 0.4297851448645815, 5.666666666666667]
Label values: [0. 1.]
Processing label 1.0
Resampling from (367, 489, 34) to [384, 512, 30]
Spacing from (0.44999998807907104, 0.44999998807907104, 5.0) to [0.4300781136068205, 0.4297851448645815, 5.666666666666667]
